In [1]:
from pyspark.sql import functions as F

# 1. Odczyt surowej tabeli Delta z Lakehouse_Bronze
bronze_fx_path = "abfss://2f59ccc2-e9d6-44b5-8c0c-1bdbf1fe148a@onelake.dfs.fabric.microsoft.com/33b93b7e-0899-43c6-81a8-654bc740aa32/Tables/dbo/fx_usd_eur_bronze"

df_raw = spark.read.format("delta").load(bronze_fx_path)

# 2. Czyszczenie, rzutowanie typów i usunięcie pustych rekordów/kolumn technicznych
df_silver = (
    df_raw
    .select(
        F.to_date(F.col("TIME_PERIOD"), "yyyy-MM-dd").alias("rate_date"),
        F.year(F.to_date(F.col("TIME_PERIOD"), "yyyy-MM-dd")).alias("rate_year"),
        F.month(F.to_date(F.col("TIME_PERIOD"), "yyyy-MM-dd")).alias("rate_month"),
        F.trim(F.col("CURRENCY")).alias("currency"),
        F.trim(F.col("CURRENCY_DENOM")).alias("currency_denom"),
        F.trim(F.col("FREQ")).alias("frequency"),
        F.col("OBS_VALUE").cast("double").alias("exchange_rate"),
        F.trim(F.col("OBS_STATUS")).alias("obs_status")
    )
    .filter(
        F.col("rate_date").isNotNull() &
        F.col("exchange_rate").isNotNull() &
        (F.col("exchange_rate") > 0)
    )
    .dropDuplicates(["rate_date", "currency", "currency_denom"])
    .orderBy(F.col("rate_date").desc())
)

# 3. Zapis oczyszczonej tabeli Delta do domyślnego Lakehouse_Silver
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("fx_usd_eur_silver")
)

print(f"Zapisano tabelę fx_usd_eur_silver. Liczba rekordów: {df_silver.count()}")
display(df_silver.limit(10))

StatementMeta(, 4036cba1-3a79-4575-bdc7-5c5eae114ce5, 3, Finished, Available, Finished, False)

Zapisano tabelę fx_usd_eur_silver. Liczba rekordów: 7133


SynapseWidget(Synapse.DataFrame, 12cabf5d-96a6-4fd6-9d47-9cf44041f766)